In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

smear_gate = nn.Linear(24, 1, bias=False)
smear_lambda = torch.tensor(torch.ones(1))

x = torch.randn(8, 2048, 32)
print(f"{x.shape=}")
x[0:4,:6,0]

x.shape=torch.Size([8, 2048, 32])


tensor([[ 0.3559, -0.8371,  0.7281,  1.0311,  1.0414,  0.7165],
        [-1.0402, -0.4798,  1.0633, -0.1238,  0.1157, -0.8498],
        [ 1.3216, -0.3641,  0.5479,  0.1773, -0.8920,  2.0771],
        [ 1.0996, -0.7609, -0.9936, -0.4376, -0.0597,  0.7381]])

In [60]:
x_current = x[:,1:]
print(f"{x_current.shape=}")  # B,T-1,C
x_current[0:4,:6,0]

x_current.shape=torch.Size([8, 2047, 32])


tensor([[-0.8371,  0.7281,  1.0311,  1.0414,  0.7165,  1.3032],
        [-0.4798,  1.0633, -0.1238,  0.1157, -0.8498, -1.4220],
        [-0.3641,  0.5479,  0.1773, -0.8920,  2.0771, -0.2019],
        [-0.7609, -0.9936, -0.4376, -0.0597,  0.7381,  1.1215]])

In [ ]:
x_score = F.sigmoid(smear_gate(x_current[...,:24]))  # B,T-1,1
print(f"{x_score.shape=}")
x_score[0:4,:6,0]

x_score.shape=torch.Size([8, 2047, 1])


tensor([[0.3467, 0.6178, 0.4232, 0.5918, 0.5076, 0.6047],
        [0.4632, 0.6391, 0.6256, 0.2925, 0.4691, 0.5629],
        [0.3003, 0.4086, 0.3926, 0.3523, 0.4966, 0.5853],
        [0.4500, 0.4810, 0.1929, 0.6805, 0.2555, 0.4901]],
       grad_fn=<SelectBackward0>)

In [63]:
x_prev_weighted = x[:,:-1] * x_score  # B,T-1,1
print(f"{x_prev_weighted.shape=}")
x_prev_weighted[0:4,:6,0]

x_prev_weighted.shape=torch.Size([8, 2047, 32])


tensor([[ 0.1234, -0.5171,  0.3081,  0.6102,  0.5286,  0.4333],
        [-0.4818, -0.3067,  0.6652, -0.0362,  0.0543, -0.4784],
        [ 0.3969, -0.1488,  0.2151,  0.0625, -0.4429,  1.2157],
        [ 0.4949, -0.3660, -0.1917, -0.2978, -0.0153,  0.3617]],
       grad_fn=<SelectBackward0>)

In [64]:
x_current_smeared = x_prev_weighted + x_current
print(f"{x_current_smeared.shape=}")
x_current_smeared[0:4,:6,0]

x_current_smeared.shape=torch.Size([8, 2047, 32])


tensor([[-0.7137,  0.2110,  1.3392,  1.6516,  1.2451,  1.7365],
        [-0.9617,  0.7566,  0.5414,  0.0795, -0.7955, -1.9004],
        [ 0.0328,  0.3991,  0.3925, -0.8295,  1.6341,  1.0138],
        [-0.2660, -1.3596, -0.6292, -0.3574,  0.7228,  1.4832]],
       grad_fn=<SelectBackward0>)

In [75]:
output = torch.cat([x[:,0:1], x_current_smeared], dim=1)
print(f"{output.shape=}")
output[0:4,:6,0]

output.shape=torch.Size([8, 2048, 32])


tensor([[ 0.3559, -0.7137,  0.2110,  1.3392,  1.6516,  1.2451],
        [-1.0402, -0.9617,  0.7566,  0.5414,  0.0795, -0.7955],
        [ 1.3216,  0.0328,  0.3991,  0.3925, -0.8295,  1.6341],
        [ 1.0996, -0.2660, -1.3596, -0.6292, -0.3574,  0.7228]],
       grad_fn=<SelectBackward0>)